In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ml-contest-cuet/sample_submission.csv
/kaggle/input/ml-contest-cuet/train.csv
/kaggle/input/ml-contest-cuet/test.csv


In [2]:
!pip install -q unsloth bitsandbytes accelerate peft transformers

# ✅ Step 2: Imports
import pandas as pd, re
from transformers import BitsAndBytesConfig
from unsloth import FastLanguageModel
import torch



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 59.5 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 10.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 5.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 6.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/tmp/ipykernel_35/1701082051.py:6: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-07-05 05:14:23.576994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751692463.807956      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751692463.869837      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:

# ✅ Step 3: Load Qwen‑3‑14B‑Instruct with 4‑bit quantization and CPU‑offload
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model, tokenizer = FastLanguageModel.from_pretrained(
   model_name= "unsloth/Qwen3-14B-Base-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()


# ✅ Step 4: Load test data
test_df = pd.read_csv("/kaggle/input/ml-contest-cuet/test.csv")



Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.6.12: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device does not support bfloat16. Will change to float16.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
import re, torch

# ──────────────────────────────────────────────
# 1️⃣  Helper: convert  "['opt1''opt2'...]"  →  list
# ──────────────────────────────────────────────
def parse_options(opt_str: str) -> list[str]:
    fixed = (opt_str.replace("''", "' '")
                     .replace('" "', '"|"')
                     .replace("'",  '"'))
    parts = re.findall(r'"([^"]+)"', fixed)
    return parts if len(parts) == 4 else opt_str.split()[:4]

# ──────────────────────────────────────────────
# 2️⃣  Helper: build a **single-question prompt**
# ──────────────────────────────────────────────
def build_prompt(question: str, options: list[str]) -> str:
    opt_block = "\n".join(f"{chr(65+i)}. {opt.strip()}" for i, opt in enumerate(options))
    return (
        f"You are a highly accurate Physics tutor.\n"
        f"You must not guess. Use short step-by-step reasoning.\n"
        f"Always select one of the four options: A, B, C, or D.\n"
        
        f"\nQuestion (Bengali): {question.strip()}\n"
        f"Options:\n{opt_block}\n\n"
        f"Write the reasoning in English.\n"
        f"At the end, write exactly one line like:\n"
        f"Answer: X   (X = A/B/C/D)\n"
        f"### Response:\n"
    )



# ──────────────────────────────────────────────
# 3️⃣  Inference loop – **one Q at a time**
# ──────────────────────────────────────────────
predictions, cots = [], []

for idx, row in test_df.iterrows():

    # ---- prepare data -------------------------------------------------
    question = row["question"]
    raw_opts = row["options"]
    options  = parse_options(raw_opts) if isinstance(raw_opts, str) else list(raw_opts)
    if len(options) != 4:
        print(f"⚠️  Q{idx}: {len(options)} options found – padded to 4.")
        options = (options + [""]*4)[:4]

    # ---- build prompt & run model -------------------------------------
    prompt   = build_prompt(question, options)
    inputs   = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=200 ,      # enough for CoT + answer
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # ---- isolate ONLY the model's new text ----------------------------
    decoded_full = tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()
    prompt_only  = tokenizer.decode(inputs["input_ids"][0],
                                    skip_special_tokens=True).strip()
    generated = decoded_full.replace(prompt_only, "", 1).strip()

    # ---- split CoT and final answer -----------------------------------
    ans_match = re.search(r"Answer[:\s]+([A-D])", generated)
    answer    = ans_match.group(1) if ans_match else "A"
    cot_text  = re.sub(r"Answer[:\s]+[A-D]\s*", "", generated).strip()

    # ---- store + display ----------------------------------------------
    predictions.append(answer)
    cots.append(cot_text)

    print("\n" + "━"*48)
    print(f"📘 Question: {question.strip()}")
    for i, o in enumerate(options):
        print(f"  {chr(65+i)}. {o.strip()}")
    print("\n🧠 Chain-of-Thought:")
    for line in cot_text.splitlines():
        if line.strip():
            print("  🔹", line.strip())
    print(f"✅ Final Answer: {answer}")

# ──────────────────────────────────────────────
# 4️⃣  `predictions` now holds one answer per row
# ──────────────────────────────────────────────



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📘 Question: কোনো বস্তুর চৌম্বকত্ব ধারকত্ব পরিমাপ করা হয়-
  A. চুম্বকনকারি বলা হয়
  B. সম্পৃক্ত দ্বারা
  C. আবিষ্ট চুম্বকত্ব দ্বারা
  D. উপরের কোনোটিই নয়

🧠 Chain-of-Thought:
  🔹 To measure the magnetic permeability of an object, we need to determine how easily it can be magnetized. Magnetic permeability is a measure of the ability of a material to support the formation of a magnetic field within itself. It is typically measured using a device called a permeameter, which applies a magnetic field to the material and measures the resulting magnetic flux density.
  🔹 Given the options:
  🔹 A. চুম্বকনকারি বলা হয় (Magnetization force)
  🔹 B. সম্পৃক্ত দ্বারা (By saturation)
  🔹 C. আবিষ্ট চুম্বকত্ব দ্বারা (By induced magnetism)
  🔹 D. উপরের কোনোটিই নয় (None of the above)
  🔹 The correct method to measure magnetic perme
✅ Final Answer: A


In [ ]:
# ✅ Step 7: Prepare submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": predictions
})
submission.to_csv("submission.csv", index=False)
print("\n✅ submission.csv created with", len(submission), "rows")